# 21 — pyFAI Handoff: convert between MIDAS spec and PoNI / `AzimuthalIntegrator`

For users migrating from pyFAI: `midas_integrate_v2.compat.pyfai` provides
bidirectional conversion between MIDAS `IntegrationSpec` and pyFAI's PoNI
calibration file (or a live `AzimuthalIntegrator`).

| function | direction |
|---|---|
| `bc_to_poni(spec)` | MIDAS → PoNI dict |
| `poni_to_bc(poni_dict, NrPixelsY, NrPixelsZ, pxY, ...)` | PoNI → MIDAS BC/Lsd/tilts |
| `make_pyfai_integrator(spec)` | live `AzimuthalIntegrator` for side-by-side runs |

In [ ]:
import os; os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import math, numpy as np, torch, matplotlib.pyplot as plt

In [ ]:
# MIDAS spec -> PoNI dict
from midas_integrate_v2 import IntegrationSpec
from midas_integrate_v2.compat import bc_to_poni, poni_to_bc, make_pyfai_integrator

t = lambda x: torch.as_tensor(x, dtype=torch.float64)
spec = IntegrationSpec()
spec.NrPixelsY = 2048; spec.NrPixelsZ = 2048
spec.pxY = 200.0; spec.pxZ = 200.0
spec.Lsd = t(940000.0); spec.BC_y = t(1024.7); spec.BC_z = t(994.3)
spec.tx = t(-0.08); spec.ty = t(-0.26); spec.tz = t(0.01)
spec.Wavelength = t(0.189714)

poni = bc_to_poni(spec)
print('PoNI fields:')
for k, v in poni.items(): print(f'  {k} = {v}')

In [ ]:
# Round-trip back: PoNI -> spec BC, Lsd, tilts
bc_y, bc_z, Lsd, tx, ty, tz = poni_to_bc(
    poni, NrPixelsY=spec.NrPixelsY, NrPixelsZ=spec.NrPixelsZ,
    pxY=spec.pxY, pxZ=spec.pxZ,
)
print(f'roundtrip diff: dBC=({bc_y-float(spec.BC_y):+.2e},{bc_z-float(spec.BC_z):+.2e}) '
      f'dLsd={Lsd-float(spec.Lsd):+.2e}')

## Run pyFAI as a sanity-check integrator

`make_pyfai_integrator(spec)` returns a live `pyFAI.AzimuthalIntegrator`
configured to match your MIDAS spec. Run pyFAI on the same image, plot
both profiles overlaid — they should agree at the 0.1 % level for a
well-calibrated geometry. Larger discrepancies indicate either a
geometry-convention mismatch or a binning convention difference (pyFAI
uses bin **centres**, MIDAS uses bin **edges**; the helper handles that).

In [ ]:
try:
    ai = make_pyfai_integrator(spec)
    print('pyFAI integrator built:', ai)
except ImportError:
    print('pyFAI not installed -- pip install pyFAI to run live')